# Analisis Clustering Ukuran Benih Lele
Notebook ini memuat ringkasan cluster, evaluasi kualitas clustering, dan interpretasi hasil berdasarkan seluruh bounding box anotasi ground truth pada dataset `train`, `valid`, dan `test`.


In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from project_helpers import (
    DATASET_ANALYSIS_DIR,
    EVALUATION_DIR,
    MODEL_ARTIFACTS_DIR,
    build_cluster_interpretation,
    build_cluster_summary,
    build_feature_matrix,
    evaluate_clustering,
)

CLASSIFIED_PATH = DATASET_ANALYSIS_DIR / "classified_sizes_all_dataset.csv"
SCALER_PATH = MODEL_ARTIFACTS_DIR / "scaler_kmeans.pkl"
KMEANS_PATH = MODEL_ARTIFACTS_DIR / "kmeans_size.pkl"

for path in [CLASSIFIED_PATH, SCALER_PATH, KMEANS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Artefak tidak ditemukan: {path}")

df = pd.read_csv(CLASSIFIED_PATH)
feature_df, X = build_feature_matrix(df)
scaler = joblib.load(SCALER_PATH)
kmeans = joblib.load(KMEANS_PATH)
X_scaled = scaler.transform(X)
df["cluster"] = kmeans.predict(X_scaled)

if "size_class" not in df.columns:
    cluster_means = feature_df.assign(cluster=df["cluster"]).groupby("cluster")["area_px"].mean().sort_values()
    mapping = {
        cluster_means.index[0]: "Fries",
        cluster_means.index[1]: "Fingerling",
        cluster_means.index[2]: "Juvenile",
    }
    df["size_class"] = df["cluster"].map(mapping)

print(f"Classified dataset file: {CLASSIFIED_PATH}")
print(f"Evaluation folder: {EVALUATION_DIR}")
print(f"Jumlah deteksi: {len(df)}")
print(f"Cluster unik: {sorted(df['cluster'].unique())}")
if "source_split" in df.columns:
    display(
        df.groupby("source_split")
        .size()
        .rename("detection_count")
        .reset_index()
    )
display(df.head())


In [ ]:
cluster_summary = build_cluster_summary(df)
interpretation_lines = build_cluster_interpretation(cluster_summary)

display(cluster_summary)
print("Interpretasi hasil clustering:")
for line in interpretation_lines:
    print("-", line)


In [ ]:
clustering_metrics = evaluate_clustering(X_scaled, df["cluster"].to_numpy())
display(clustering_metrics)

metric_row = clustering_metrics.iloc[0]
silhouette = metric_row["silhouette_score"]
db_index = metric_row["davies_bouldin_score"]
ch_index = metric_row["calinski_harabasz_score"]

if silhouette >= 0.5:
    silhouette_note = "pemisahan antarkelompok kuat"
elif silhouette >= 0.25:
    silhouette_note = "pemisahan cluster cukup baik"
else:
    silhouette_note = "pemisahan cluster masih lemah"

print(f"Silhouette score = {silhouette:.3f}, artinya {silhouette_note}.")
print(f"Davies-Bouldin score = {db_index:.3f}; semakin kecil nilainya semakin baik.")
print(f"Calinski-Harabasz score = {ch_index:.1f}; semakin besar nilainya semakin baik.")


In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    df["width_px"],
    df["height_px"],
    c=df["cluster"],
    cmap="tab10",
    alpha=0.7,
)

plt.xlabel("Bounding Box Width (pixel)")
plt.ylabel("Bounding Box Height (pixel)")
plt.title("Visualisasi Distribusi Cluster Ukuran Benih Lele")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plot_order = cluster_summary["size_class"].tolist()

plt.figure(figsize=(8, 6))
df.boxplot(column="area_px", by="size_class", grid=True)
plt.xlabel("Size Class")
plt.ylabel("Bounding Box Area (pixel2)")
plt.title("Distribusi Ukuran Benih Lele Berdasarkan Cluster")
plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))

for cluster_id in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == cluster_id]
    label = subset["size_class"].mode().iloc[0]
    plt.hist(
        subset["area_px"],
        bins=20,
        alpha=0.5,
        label=f"Cluster {cluster_id} - {label}",
    )

plt.xlabel("Bounding Box Area (pixel2)")
plt.ylabel("Jumlah Sampel")
plt.title("Histogram Distribusi Ukuran Benih Lele per Cluster")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
